In [1]:
!pip install requests pandas beautifulsoup4

In [5]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [7]:
locations = {
    "Bangalore Central": (12.9716, 77.5946),
    "Whitefield": (12.9698, 77.7500),
    "Electronic City": (12.8452, 77.6602),
    "Yelahanka": (13.1007, 77.5963),
    "Jayanagar": (12.9250, 77.5938)
}

locations

{'Bangalore Central': (12.9716, 77.5946),
 'Whitefield': (12.9698, 77.75),
 'Electronic City': (12.8452, 77.6602),
 'Yelahanka': (13.1007, 77.5963),
 'Jayanagar': (12.925, 77.5938)}

In [8]:
weather_url = "https://api.open-meteo.com/v1/forecast"

weather_data = []

for location, (latitude, longitude) in locations.items():

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,relative_humidity_2m",
        "past_days": 21,
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(weather_url, params=params)

    print(location, "Status:", response.status_code)

    if response.status_code == 200:
        data = response.json()

        df = pd.DataFrame({
            "location": location,
            "time": data["hourly"]["time"],
            "temperature": data["hourly"]["temperature_2m"],
            "humidity": data["hourly"]["relative_humidity_2m"]
        })

        weather_data.append(df)

    time.sleep(1)

weather_df = pd.concat(weather_data, ignore_index=True)

weather_df.head()

Bangalore Central Status: 200
Whitefield Status: 200
Electronic City Status: 200
Yelahanka Status: 200
Jayanagar Status: 200


,location,time,temperature,humidity
0,Bangalore Central,2026-07-30T00:00,22.1,88
1,Bangalore Central,2026-07-30T01:00,21.9,90
2,Bangalore Central,2026-07-30T02:00,21.5,88
3,Bangalore Central,2026-07-30T03:00,21.2,90
4,Bangalore Central,2026-07-30T04:00,21.0,90


In [9]:
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

air_quality_data = []

for location, (latitude, longitude) in locations.items():

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "pm10,pm2_5,carbon_monoxide",
        "past_days": 30,
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(aq_url, params=params)

    print(location, "Status:", response.status_code)

    if response.status_code == 200:
        data = response.json()

        df = pd.DataFrame({
            "location": location,
            "time": data["hourly"]["time"],
            "pm10": data["hourly"]["pm10"],
            "pm2_5": data["hourly"]["pm2_5"],
            "carbon_monoxide": data["hourly"]["carbon_monoxide"]
        })

        air_quality_data.append(df)

    time.sleep(1)

air_quality_df = pd.concat(air_quality_data, ignore_index=True)

air_quality_df.head()

Bangalore Central Status: 200
Whitefield Status: 200
Electronic City Status: 200
Yelahanka Status: 200
Jayanagar Status: 200


,location,time,pm10,pm2_5,carbon_monoxide
0,Bangalore Central,2026-07-21T00:00,5.6,4.5,233.0
1,Bangalore Central,2026-07-21T01:00,4.7,3.8,186.0
2,Bangalore Central,2026-07-21T02:00,4.6,3.6,158.0
3,Bangalore Central,2026-07-21T03:00,5.0,3.7,148.0
4,Bangalore Central,2026-07-21T04:00,5.6,4.0,155.0


In [10]:
air_quality_df.to_csv("bengaluru_air_quality.csv", index=False)

print("Air quality data saved successfully!")
print("Number of records:", len(air_quality_df))
print("Columns:", air_quality_df.columns.tolist())

Air quality data saved successfully!
Number of records: 4200
Columns: ['location', 'time', 'pm10', 'pm2_5', 'carbon_monoxide']


In [11]:
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

books = []

page = 1

while True:

    url = base_url.format(page)

    response = requests.get(url)

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    book_items = soup.select("article.product_pod")

    if not book_items:
        break

    for book in book_items:

        title = book.h3.a["title"]

        price = book.select_one(".price_color").text.strip()

        rating = book.select_one(".star-rating")["class"][1]

        books.append({
            "title": title,
            "price": price,
            "rating": rating
        })

    print("Page", page, "collected")

    page += 1

    time.sleep(1)

books_df = pd.DataFrame(books)

books_df.head()

Page 1 collected
Page 2 collected
Page 3 collected
Page 4 collected
Page 5 collected
Page 6 collected
Page 7 collected
Page 8 collected
Page 9 collected
Page 10 collected
Page 11 collected
Page 12 collected
Page 13 collected
Page 14 collected
Page 15 collected
Page 16 collected
Page 17 collected
Page 18 collected
Page 19 collected
Page 20 collected
Page 21 collected
Page 22 collected
Page 23 collected
Page 24 collected
Page 25 collected
Page 26 collected
Page 27 collected
Page 28 collected
Page 29 collected
Page 30 collected
Page 31 collected
Page 32 collected
Page 33 collected
Page 34 collected
Page 35 collected
Page 36 collected
Page 37 collected
Page 38 collected
Page 39 collected
Page 40 collected
Page 41 collected
Page 42 collected
Page 43 collected
Page 44 collected
Page 45 collected
Page 46 collected
Page 47 collected
Page 48 collected
Page 49 collected
Page 50 collected


,title,price,rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five


In [12]:
books_df.to_csv("books_data.csv", index=False)

print("Book data saved successfully!")
print("Number of books:", len(books_df))
print("Columns:", books_df.columns.tolist())

Book data saved successfully!
Number of books: 1000
Columns: ['title', 'price', 'rating']


In [13]:
print("Weather file:")
print(weather_df.head())

print("\nAir Quality file:")
print(air_quality_df.head())

print("\nBookstore file:")
print(books_df.head())

Weather file:
            location              time  temperature  humidity
0  Bangalore Central  2026-07-30T00:00         22.1        88
1  Bangalore Central  2026-07-30T01:00         21.9        90
2  Bangalore Central  2026-07-30T02:00         21.5        88
3  Bangalore Central  2026-07-30T03:00         21.2        90
4  Bangalore Central  2026-07-30T04:00         21.0        90

Air Quality file:
            location              time  pm10  pm2_5  carbon_monoxide
0  Bangalore Central  2026-07-21T00:00   5.6    4.5            233.0
1  Bangalore Central  2026-07-21T01:00   4.7    3.8            186.0
2  Bangalore Central  2026-07-21T02:00   4.6    3.6            158.0
3  Bangalore Central  2026-07-21T03:00   5.0    3.7            148.0
4  Bangalore Central  2026-07-21T04:00   5.6    4.0            155.0

Bookstore file:
                                   title    price rating
0                   A Light in the Attic  Â£51.77  Three
1                     Tipping the Velvet  Â£53.74 

In [14]:
print("DATA COLLECTION COMPLETED")
print("-------------------------")

print("Weather records:", len(weather_df))
print("Weather columns:", weather_df.columns.tolist())

print("\nAir quality records:", len(air_quality_df))
print("Air quality columns:", air_quality_df.columns.tolist())

print("\nBook records:", len(books_df))
print("Book columns:", books_df.columns.tolist())

print("\nFiles created:")
print("1. bengaluru_weather.csv")
print("2. bengaluru_air_quality.csv")
print("3. books_data.csv")


DATA COLLECTION COMPLETED
-------------------------
Weather records: 3360
Weather columns: ['location', 'time', 'temperature', 'humidity']

Air quality records: 4200
Air quality columns: ['location', 'time', 'pm10', 'pm2_5', 'carbon_monoxide']

Book records: 1000
Book columns: ['title', 'price', 'rating']

Files created:
1. bengaluru_weather.csv
2. bengaluru_air_quality.csv
3. books_data.csv
